In [1]:
import duckdb
import pandas as pd 

In [2]:
con = duckdb.connect()
con.execute("""
    CREATE TABLE  employees AS
    SELECT * FROM read_csv_auto('../data/raw/employees.csv')
""")
con.execute("""
    CREATE TABLE positions AS
    SELECT * FROM read_csv_auto('../data/raw/positions.csv')
""")
con.execute("""
    CREATE TABLE hr_tickets AS
    SELECT * FROM read_csv_auto('../data/raw/hr_tickets.csv')
""")
print("Tables loaded.")

Tables loaded.


In [3]:
for table in ['employees', 'positions', 'hr_tickets']:
    count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}: {count} rows")

employees: 427 rows
positions: 27 rows
hr_tickets: 2800 rows


In [4]:
result = con.execute("""
    SELECT COUNT(*) AS orphan_count
    FROM hr_tickets t
    LEFT JOIN positions p ON t.position_id = p.position_id
    WHERE p.position_id is NULL
""").df()
print("Orphan position IDs in hr_tickets:")
result

Orphan position IDs in hr_tickets:


,orphan_count
0,1977


In [5]:
result = con.execute("""
    SELECT first_name, last_name, COUNT(*) AS occurrences
    FROM employees
    GROUP BY first_name, last_name
    HAVING COUNT(*) > 1
    ORDER BY occurrences DESC
""").df()
print(f"Employees with duplicate names: {len(result)}")
result

Employees with duplicate names: 8


,first_name,last_name,occurrences
0,Daniel,Johnson,2
1,Elizabeth,Mcmillan,2
2,Joseph,Bowers,2
3,Linda,Wolfe,2
4,Rodney,Hill,2
5,Collin,Jordan,2
6,George,Daniel,2
7,James,Moran,2


In [6]:
result = con.execute("""
    SELECT position_id, COUNT(DISTINCT effective_date) AS version_count
    FROM positions
    GROUP BY position_id
    HAVING COUNT(DISTINCT effective_date) > 1
    ORDER BY version_count DESC
""").df()
print(f"Positions with multiple versions: {len(result)}")
result

Positions with multiple versions: 3


,position_id,version_count
0,SALES-IC2,2
1,CS-IC2,2
2,ENG-IC1,2


In [7]:
result = con.execute("""
    SELECT category, COUNT(*) as ticket_count
    FROM hr_tickets
    GROUP BY category
    ORDER BY ticket_count DESC
""").df()
print("Category values in hr_tickets:")
result

Category values in hr_tickets:


,category,ticket_count
0,HR Technology,665
1,Payroll,521
2,Policy & Benefits,427
3,Onboarding,346
4,Learning & Development,290
5,Offboarding,191
6,Employee Relations,188
7,Compensation,135
8,HR Tech,20
9,Benefits,6


In [8]:
result = con.execute("""
    SELECT country, COUNT(*) AS employee_count
    FROM employees
    GROUP BY country
    ORDER BY employee_count DESC
""").df()
print("Country values in employees:")
result

Country values in employees:


,country,employee_count
0,US,275
1,UK,67
2,CA,25
3,U.S.,21
4,United States,20
5,USA,9
6,GB,4
7,Canada,3
8,England,1
9,CAN,1


In [9]:
result = con.execute("""
    SELECT COUNT(*) AS invalid_tickets
    FROM hr_tickets
    WHERE close_date < open_date
""").df()
print("Tickets with close date before open date:")
result

Tickets with close date before open date:


,invalid_tickets
0,72


In [10]:
result = con.execute("""
    SELECT dept,
        COUNT(*) AS total_employees,
            SUM(CASE WHEN manager_id IS NULL THEN 1 ELSE 0 END) AS null_manager_count
    FROM employees                 
    GROUP BY dept
    ORDER BY null_manager_count DESC                 
""").df()
print("NULL manager_id by department:")
result

NULL manager_id by department:


,dept,total_employees,null_manager_count
0,Engineering,187,18.0
1,Customer Success,68,13.0
2,Product,41,4.0
3,Sales,56,4.0
4,Data,37,4.0
5,Executive,3,3.0
6,HR,11,1.0
7,Finance,10,1.0
8,Marketing,14,0.0


In [11]:
result = con.execute("""
    SELECT e.employee_id, e.first_name, e.last_name, e.dept,
        e.salary, p.band_min, p.band_max
    FROM employees e
    JOIN positions p ON e.position_id = p.position_id
    WHERE e.salary < p.band_min OR e.salary  > p.band_max
    ORDER BY e.dept
""").df()
print(f"Employees paid outside their salary band: {len(result)}")
result

Employees paid outside their salary band: 45


,employee_id,first_name,last_name,dept,salary,band_min,band_max
0,EMP-00331,Anna,Torres,Customer Success,51166,91800,118800
1,EMP-00328,Lynn,Morales,Customer Success,86500,91800,118800
2,EMP-00326,Jack,Williams,Customer Success,85000,91800,118800
3,EMP-00318,James,Perez,Customer Success,86500,91800,118800
4,EMP-00312,James,Wilson,Customer Success,86500,91800,118800
5,EMP-00313,Amanda,Gill,Customer Success,85500,91800,118800
6,EMP-00334,Krystal,Perez,Customer Success,86500,91800,118800
7,EMP-00323,Miranda,Henry,Customer Success,85500,91800,118800
8,EMP-00315,Todd,Hardy,Customer Success,85000,91800,118800
9,EMP-00332,Victoria,Wilson,Customer Success,86500,91800,118800


In [12]:
employees = con.execute("SELECT * FROM employees").df()
positions = con.execute("SELECT * FROM positions").df()
hr_tickets = con.execute("SELECT * FROM hr_tickets").df()

print(f"employees: {len(employees)} rows")
print(f"positions: {len(positions)} rows")
print(f"hr_tickets: {len(hr_tickets)} rows")

employees: 427 rows
positions: 27 rows
hr_tickets: 2800 rows


In [13]:
employees_clean = employees.sort_values('hire_date', ascending=False)
employees_clean = employees_clean.drop_duplicates(subset=['first_name', 'last_name'], keep='first')

print(f"Before: {len(employees)} rows")
print(f"After: {len(employees_clean)} rows")
print(f"Removed: {len(employees) - len(employees_clean)} duplicates")

Before: 427 rows
After: 419 rows
Removed: 8 duplicates


In [14]:
positions_clean = positions.sort_values('effective_date', ascending=False)
positions_clean = positions_clean.drop_duplicates(subset=['position_id'], keep='first')

print(f"Before: {len(positions)} rows")
print(f"After: {len(positions_clean)} rows")
print(f"Removed: {len(positions) - len(positions_clean)} old versions")

Before: 27 rows
After: 24 rows
Removed: 3 old versions


In [15]:
valid_position_ids = set(positions_clean['position_id'])

hr_tickets_clean = hr_tickets.copy()
hr_tickets_clean['position_id'] = hr_tickets_clean['position_id'].apply(
    lambda x: x if x in valid_position_ids else 'Unknown-Role'
)
orphan_count = (hr_tickets_clean['position_id'] == 'Unknown-Role').sum()
print(f"Tickets tagged as Unknown-Role: {orphan_count}")


Tickets tagged as Unknown-Role: 1977


In [16]:
category_map = {
    'HR Tech' : 'HR Technology',
    'Benefits' : 'Policy & Benefits',
    'L&D' : 'Learning & Development',
    'ER' : 'Employee Relations'    
}
hr_tickets_clean['category'] = hr_tickets_clean['category'].replace(category_map)
legacy_remaining = hr_tickets_clean['category'].isin(category_map.keys()).sum()
print(f"Legacy category values remaining: {legacy_remaining}")

Legacy category values remaining: 0


In [17]:
country_map = {
    'U.S.': 'US',
    'United States': 'US',
    'USA': 'US',
    'UK': 'GB',
    'United Kingdom': 'GB',
    'England': 'GB',
    'Canada': 'CA',
    'CAN': 'CA'
}

employees_clean['country'] = employees_clean['country'].replace(country_map)

print("Country values after normalization:")
employees_clean['country'].value_counts()

Country values after normalization:


country
US    321
GB     70
CA     28
Name: count, dtype: int64

In [18]:
hr_tickets_clean = hr_tickets_clean[
    (hr_tickets_clean['close_date'] >= hr_tickets_clean['open_date']) |
    (hr_tickets_clean['close_date'].isna())
]

print(f"Before: {len(hr_tickets)} rows")
print(f"After: {len(hr_tickets_clean)} rows")
print(f"Dropped: {len(hr_tickets) - len(hr_tickets_clean)} invalid tickets")

Before: 2800 rows
After: 2728 rows
Dropped: 72 invalid tickets


In [19]:
null_managers = employees_clean['manager_id'].isna().sum()
print(f"Employees with NULL manager_id: {null_managers}")
print("Decision: keeping NULLs as-is - org gaps are a data quality finding, nor missing data to fill.")

Employees with NULL manager_id: 48
Decision: keeping NULLs as-is - org gaps are a data quality finding, nor missing data to fill.


In [21]:
employees_clean['salary_outlier'] = False
for dept in employees_clean['dept'].unique() :
    mask = employees_clean['dept'] == dept
    dept_salaries = employees_clean.loc[mask, 'salary']

    Q1 = dept_salaries.quantile(0.25)
    Q3 = dept_salaries.quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outlier_mask = mask & ((employees_clean['salary'] < lower) | (employees_clean['salary'] > upper))
    employees_clean.loc[outlier_mask, 'salary_outlier'] = True

flagged = employees_clean['salary_outlier'].sum()
print(f"Employees flagged as salary outliers: {flagged}")

Employees flagged as salary outliers: 16


In [22]:
con.execute("CREATE OR REPLACE TABLE employees_clean AS SELECT * FROM employees_clean")
con.execute("CREATE OR REPLACE TABLE positions_clean AS SELECT * FROM positions_clean")
con.execute("CREATE OR REPLACE TABLE hr_tickets_clean AS SELECT * FROM hr_tickets_clean")

print("Clean tables written to DuckDB.")

Clean tables written to DuckDB.


In [23]:
import os
os.makedirs('../data/clean', exist_ok = True)

employees_clean.to_csv('../data/clean/employees.csv', index = False)
positions_clean.to_csv('../data/clean/positions.csv', index = False)
hr_tickets_clean.to_csv('../data/clean/hr_tickets.csv', index = False)

print("Exported:")
print(f"  employees:  {len(employees_clean)} rows → data/clean/employees.csv")
print(f"  positions:  {len(positions_clean)} rows → data/clean/positions.csv")
print(f"  hr_tickets: {len(hr_tickets_clean)} rows → data/clean/hr_tickets.csv")

Exported:
  employees:  419 rows → data/clean/employees.csv
  positions:  24 rows → data/clean/positions.csv
  hr_tickets: 2728 rows → data/clean/hr_tickets.csv


In [24]:
clean_position_ids = set(positions_clean['position_id'])

orphans_remaining = hr_tickets_clean[
    (hr_tickets_clean['position_id'] != 'Unknown-Role') &
    (~hr_tickets_clean['position_id'].isin(clean_position_ids))
]

print(f"Orphan FKs remaining in clean hr_tickets: {len(orphans_remaining)}")

Orphan FKs remaining in clean hr_tickets: 0
